<center>
<h1><b>Information Retrieval : RAG (Retrieval Augmented Generation)</b></h1>
<h3>TP — Analyse de CV avec un système RAG</h3>
</center>

## Introduction

Jusqu'à présent, les LLM s'appuient sur les connaissances internalisées lors de leur pré-entraînement. Cela pose un problème dès lors qu'on veut interroger un document privé (un CV, un rapport, un contrat…) que le modèle n'a jamais vu.

La technique **RAG (Retrieval Augmented Generation)** résout ce problème en deux temps :
1. **Indexation** : on découpe le document en morceaux (chunks), on les encode en vecteurs (embeddings), et on les stocke dans une base vectorielle.
2. **Retrieval + Génération** : à chaque question, on retrouve les chunks les plus pertinents, puis on les injecte dans le prompt du LLM pour qu'il génère une réponse fondée sur le document.

Dans ce TP, nous allons construire un système RAG complet pour interroger un **CV** en langage naturel.

## Les blocs de construction du RAG

### Workflow global

```
Document PDF
     │
     ▼
  Chunking  ──► Embeddings ──► Vector Store (ChromaDB)
                                        │
Question utilisateur ──► Embedding ──► Retriever (top-k chunks)
                                        │
                              Prompt = contexte + question
                                        │
                                   LLM (gpt-4o-mini)
                                        │
                                    Réponse
```

### Pourquoi le chunking ?
Les modèles d'embedding ont une fenêtre de contexte limitée. On découpe donc le document en petits morceaux (chunks) avec un chevauchement (overlap) pour ne pas perdre le sens entre deux chunks consécutifs.

### Pourquoi ChromaDB ?
ChromaDB est une base de données vectorielle légère et open-source, idéale pour les prototypes. Elle permet de persister les embeddings sur disque et d'effectuer des recherches par similarité (cosine, L2…) très rapidement.

---
## Étape 0 — Imports et configuration

In [1]:
import os
import tiktoken

from dotenv import load_dotenv
from IPython.display import Markdown, display

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

In [2]:
load_dotenv(override=True)

True

In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

---
## Étape 1 — Chargement du CV et Chunking

Nous utilisons `PyPDFLoader` de LangChain pour lire le PDF page par page, puis `RecursiveCharacterTextSplitter` pour découper le texte en chunks.

**Paramètres choisis :**
- `chunk_size = 300` tokens — taille adaptée à un CV (sections courtes)
- `chunk_overlap = 20` tokens — chevauchement pour conserver la continuité

In [4]:
pdf_file = "./cv_junior_eria (1).pdf"
pdf_loader = PyPDFLoader(pdf_file)

In [5]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=300,
    chunk_overlap=20
)

In [6]:
chunks = pdf_loader.load_and_split(text_splitter)

In [7]:
# Nombre total de chunks générés
print(f"Nombre de chunks : {len(chunks)}")
print("\nAperçu du premier chunk :")
print(chunks[0].page_content)

Nombre de chunks : 5

Aperçu du premier chunk :
GATSOUNDOU 
Junior Stevy 
Ingénieur d'État IA & Big Data 
Automatisation & Agents IA 
C   O   N   T   A   C   T 
✉    j.gatsoundou@hestim.ma 
☎    +212 06 12 43 56 68 
◆    Casablanca, Maroc 
in   Junior Stevy Gatsoundou 
⌂  gatsoundoujunior-netizen   
 
 
C   O   M   P   É   T   E   N   C   E   S 
Automatisation No-Code 
n8n · Supabase · Docker · Git Actions 
Agents & LLM 
LangChain · LangGraph · Groq · OpenAI 
Machine Learning 
Scikit-learn · Random Forest · Isolation 
Forest 
Déploiement 
FastAPI · Streamlit · Flask · MongoDB 
Langages 
Python · SQL · JavaScript · HTML/CSS 
 
L   A   N   G   U   E   S 
Français — langue maternelle 
Anglais — B1, en progression 
 
C E R T I F I C A T I O N S 
✓  Google Agile Foundations 
✓  Python for Data Science & AI — IBM


---
## Étape 2 — Vector Store : ChromaDB + Embeddings

### Choix du modèle d'embedding
Nous utilisons `text-embedding-ada-002` d'OpenAI — un modèle polyvalent qui produit des vecteurs de dimension 1536. Il encode sémantiquement le texte : deux phrases de sens proche auront des vecteurs proches dans l'espace vectoriel.

### Persistance
En passant `persist_directory="./store"`, ChromaDB sauvegarde les embeddings sur disque. Au prochain lancement, on peut recharger le store sans recalculer les embeddings.

In [8]:
embedding_model = OpenAIEmbeddings(model='text-embedding-ada-002')

In [9]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="cv_junior",
    persist_directory="./store"
)

In [10]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [11]:
# Test du retriever : recherche des chunks pertinents
test_query = "Quelles sont les compétences techniques ?"
retrieved_chunks = retriever.invoke(test_query)
print(f"Nombre de chunks récupérés : {len(retrieved_chunks)}")
print("\nContenu du chunk le plus pertinent :")
print(retrieved_chunks[0].page_content)

Nombre de chunks récupérés : 5

Contenu du chunk le plus pertinent :
✓  Google Agile Foundations 
✓  Python for Data Science & AI — IBM 
○  AWS Cloud Practitioner (en cours) 
○  IBM AI Engineering (en cours) 
○  IBM RAG & Agentic AI (en cours) 
○  Azure AZ-900 (en cours) 
 
S   O   F   T     S   K   I   L   L   S 
› Autonome & rigoureux 
› Esprit analytique 
› Force de proposition 
› Mode startup 
› Travail en équipe 
 
PROFIL 
Ingénieur IA & Big Data (HESTIM Casablanca), spécialisé en automatisation et agents IA. 
Actuellement en stage chez Holokia, je conçois AutoFlow : plateforme multi -agents n8n avec 
workflows d'automatisation, cartographie et optimisation de processus, intégration d'APIs. Curieux, 
autonome, appétence entrepreneuriale forte  à l'aise en mode startup et full remote. Disponible 
pour un stage PFE de fin d'études (6 mois) à partir de janvier 2026.  
FORMATION 
Cycle Ingénieur d'État — Informatique & IA  |  2025 – 2026 
HESTIM — Casablanca


---
## Étape 3 — RAG Q&A

### Design du Prompt

Le prompt est conçu avec des instructions strictes :
- Le LLM ne doit répondre qu'à partir du contexte fourni
- Si la réponse n'est pas dans le contexte → `JE NE SAIS PAS`

Cela permet d'éviter les **hallucinations** (le LLM qui invente une réponse non fondée sur le document).

In [12]:
prompt_template = """
Réponds à la question suivante en te basant UNIQUEMENT sur le contexte fourni.
Le contexte provient d'un CV professionnel.
Le contexte est délimité par la balise <context>.
La question est délimitée par la balise <question>.
Si la réponse n'est pas dans le contexte, réponds : JE NE SAIS PAS

<context>
 {context}
</context>

<question>
 {question}
</question>
"""

### Récupération des documents pertinents

In [13]:
user_input = "Quelles sont les expériences professionnelles du candidat ?"

In [14]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [15]:
context_for_query

"HESTIM — Casablanca \nCycle Ingénieur d'État — Informatique  |  2023 – 2024 \nEMG — Rabat \nClasse Préparatoire  |  2021 – 2023 \nESSTI — Rabat \n Prix Meilleure Entreprise — Projet Intégrateur 2023-2024 \nEXPÉRIENCE  PROFESSIONNELLE \nStagiaire Automatisation & IA — Agents IA \nHolokia — Casablanca (Startup Tech) Stage en cours · jan. – juil. 2025 \n› Conçu et déployé AutoFlow : plateforme multi-agents n8n complète sous Docker (Supabase + React) de \nl'idée à la prod \n› Développé 4 workflows d'automatisation actifs : scoring CV (Groq LLaMA-3.3-70b), transcription réunions \n(Whisper), veille Hacker News, comptes rendus automatiques \n› Orchestré des agents IA  : Gmail Trigger → Text Classifier → AI Agent → Supabase → email de confirmation \n› Mis en place le DevOps : CI/CD GitHub Actions, branches main/develop, ESLint, notifications automatiques \n› Cartographié les processus métier, identifié les opportunités d'automatisation et mesuré l'impact des \nworkflows déployés \nStagiaire 

In [16]:
len(relevant_document_chunks)

5

In [17]:
prompt = prompt_template.format(context=context_for_query, question=user_input)
print(prompt)


Réponds à la question suivante en te basant UNIQUEMENT sur le contexte fourni.
Le contexte provient d'un CV professionnel.
Le contexte est délimité par la balise <context>.
La question est délimitée par la balise <question>.
Si la réponse n'est pas dans le contexte, réponds : JE NE SAIS PAS

<context>
 HESTIM — Casablanca 
Cycle Ingénieur d'État — Informatique  |  2023 – 2024 
EMG — Rabat 
Classe Préparatoire  |  2021 – 2023 
ESSTI — Rabat 
 Prix Meilleure Entreprise — Projet Intégrateur 2023-2024 
EXPÉRIENCE  PROFESSIONNELLE 
Stagiaire Automatisation & IA — Agents IA 
Holokia — Casablanca (Startup Tech) Stage en cours · jan. – juil. 2025 
› Conçu et déployé AutoFlow : plateforme multi-agents n8n complète sous Docker (Supabase + React) de 
l'idée à la prod 
› Développé 4 workflows d'automatisation actifs : scoring CV (Groq LLaMA-3.3-70b), transcription réunions 
(Whisper), veille Hacker News, comptes rendus automatiques 
› Orchestré des agents IA  : Gmail Trigger → Text Classifier → A

In [18]:
resp = llm.invoke(prompt)

In [19]:
display(Markdown(resp.content))

Le candidat a deux expériences professionnelles :

1. Stagiaire Automatisation & IA chez Holokia — Casablanca (Startup Tech) : Stage en cours depuis janvier 2025, où il conçoit et déploie AutoFlow, développe des workflows d'automatisation, orchestre des agents IA, met en place le DevOps et cartographie les processus métier.

2. Stagiaire IT au CHU Ibn Sina, Rabat (2021 – 2022) : Il a effectué la maintenance du parc informatique, l'assistance technique, le support utilisateurs et la résolution d'incidents.

### Définition de la fonction RAG

On encapsule toute la chaîne (retrieval → prompt → LLM) dans une fonction réutilisable.

In [20]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query = ". ".join(context_list)
    prompt = prompt_template.format(context=context_for_query, question=query)
    resp = llm.invoke(prompt)
    return resp.content

In [21]:
# Test 1 : formation académique
response = RAG("Quelle est la formation académique du candidat ?")
display(Markdown(response))

Le candidat a suivi les formations académiques suivantes :

1. Cycle Ingénieur d'État — Informatique & IA à HESTIM, Casablanca (2025 – 2026)
2. Cycle Ingénieur d'État — Informatique à EMG, Rabat (2023 – 2024)
3. Classe Préparatoire à ESSTI, Rabat (2021 – 2023)

In [22]:
# Test 2 : compétences techniques
response = RAG("Quels sont les langages de programmation maîtrisés par le candidat ?")
display(Markdown(response))

Les langages de programmation maîtrisés par le candidat sont : Python, SQL, JavaScript, HTML/CSS.

In [23]:
# Test 3 : question hors scope — doit répondre JE NE SAIS PAS
user_input_hors_scope = "Quel est le prix du pétrole en 2024 ?"
output = RAG(user_input_hors_scope)
print(output)

JE NE SAIS PAS


In [24]:
# Test 4 : langues
response = RAG("Quelles langues parle le candidat ?")
display(Markdown(response))

Le candidat parle français (langue maternelle) et anglais (niveau B1, en progression).

In [25]:
# Test 5 : résumé global
response = RAG("Fais un résumé complet du profil du candidat.")
display(Markdown(response))

Le candidat, Gatsoundou Junior Stevy, est un ingénieur spécialisé en IA et Big Data, diplômé de HESTIM Casablanca. Actuellement en stage chez Holokia, il travaille sur la conception d'AutoFlow, une plateforme multi-agents pour l'automatisation et l'optimisation de processus. Il possède des compétences en développement de workflows d'automatisation, orchestration d'agents IA, et mise en place de DevOps. 

Il a une formation solide en informatique et IA, avec un cycle d'ingénieur d'État en cours et une expérience antérieure en maintenance informatique et support technique au CHU Ibn Sina. Il a également participé à des projets notables, tels que la détection précoce d'anomalies portuaires et un système décisionnel pour la détection de fraude bancaire, démontrant des compétences en machine learning et en développement d'applications.

Le candidat est autonome, rigoureux, et a un esprit analytique, avec une forte appétence entrepreneuriale et une aisance en mode startup. Il est disponible pour un stage de fin d'études de 6 mois à partir de janvier 2026. Il parle français comme langue maternelle et a un niveau B1 en anglais. Il est en cours d'obtention de plusieurs certifications, dont AWS Cloud Practitioner et IBM AI Engineering.

---
## Étape 4 — Évaluation : LLM-as-a-Judge

Pour évaluer la qualité du système RAG, on utilise un **LLM juge** (ici `gpt-4o`). On évalue la métrique de **Groundedness** :

> **Groundedness** : la réponse est-elle entièrement fondée sur le contexte récupéré, sans hallucination ?

Le juge attribue un score de 1 à 5 :
- 1 = la réponse n'est pas du tout fondée sur le contexte
- 5 = la réponse est entièrement issue du contexte

In [26]:
user_input = "Quelles sont les compétences techniques du candidat ?"

In [27]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [28]:
user_message_template = """
 ###Question
 {question}
 ###Context
 {context}
 ###Answer
 {answer}
"""

In [29]:
# Réponse générée par le système RAG
answer = RAG(user_input)
display(Markdown(answer))

Les compétences techniques du candidat incluent :

- Automatisation No-Code : n8n, Supabase, Docker, Git Actions
- Agents & LLM : LangChain, LangGraph, Groq, OpenAI
- Machine Learning : Scikit-learn, Random Forest, Isolation Forest
- Déploiement : FastAPI, Streamlit, Flask, MongoDB
- Langages : Python, SQL, JavaScript, HTML/CSS

### Groundedness

In [30]:
groundedness_rater_system_message = """
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse,
ainsi qu'une réponse générée par l'IA à la question.

Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context,
et la réponse générée par l'IA commencera par ###Answer.

Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1 – La métrique n'est pas respectée du tout
2 – La métrique n'est respectée que dans une mesure limitée
3 – La métrique est respectée dans une bonne mesure
4 – La métrique est respectée en grande partie
5 – La métrique est entièrement respectée

Métrique :
La réponse doit être dérivée uniquement des informations présentes dans le contexte.

Instructions :
Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique,
en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [31]:
groundness_checker = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

In [32]:
def evaluate(system_message, user_message_template, question, model=groundness_checker):
    retrieved_chunks = retriever.invoke(question)
    context_list = [d.page_content for d in retrieved_chunks]
    context = ". ".join(context_list)
    answer = RAG(question)
    prompt = f"""
     {system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
    """
    juge_response = model.invoke(prompt)
    return juge_response.content

In [33]:
resp = evaluate(groundedness_rater_system_message, user_message_template, user_input)

In [34]:
display(Markdown(resp))

Pour évaluer la réponse selon la métrique, voici les étapes nécessaires :

1. **Identifier la question** : La question demande quelles sont les compétences techniques du candidat.

2. **Analyser le contexte** : Le contexte fournit des informations sur les compétences techniques, les certifications, les expériences professionnelles, les projets, et les outils utilisés par le candidat. Les compétences techniques sont spécifiquement listées sous la section "COMPÉTENCES".

3. **Comparer la réponse avec le contexte** : La réponse doit être vérifiée par rapport aux informations fournies dans le contexte. Les compétences techniques mentionnées dans la réponse doivent correspondre à celles listées dans le contexte.

4. **Vérification de l'exhaustivité** : S'assurer que toutes les compétences techniques mentionnées dans le contexte sont couvertes dans la réponse.

5. **Évaluer la conformité à la métrique** : La réponse doit être dérivée uniquement des informations présentes dans le contexte.

**Évaluation étape par étape :**

- La réponse mentionne les compétences techniques suivantes : Automatisation No-Code (n8n, Supabase, Docker, Git Actions), Agents & LLM (LangChain, LangGraph, Groq, OpenAI), Machine Learning (Scikit-learn, Random Forest, Isolation Forest), Déploiement (FastAPI, Streamlit, Flask, MongoDB), et Langages (Python, SQL, JavaScript, HTML/CSS).
  
- En comparant avec le contexte, toutes ces compétences sont effectivement listées sous la section "COMPÉTENCES" du contexte.

- La réponse est complète et n'omets aucune compétence technique mentionnée dans le contexte.

- La réponse ne contient pas d'informations supplémentaires qui ne seraient pas présentes dans le contexte.

**Évaluation finale :**

La réponse respecte entièrement la métrique car elle est dérivée uniquement des informations présentes dans le contexte et couvre toutes les compétences techniques mentionnées. Par conséquent, la réponse mérite un score de 5.

---
## Conclusion

Dans ce TP, nous avons construit un système **RAG complet** pour interroger un CV en langage naturel :

1. **Chargement & Chunking** : lecture du PDF avec `PyPDFLoader`, découpage en chunks de 300 tokens avec `RecursiveCharacterTextSplitter`.
2. **Indexation vectorielle** : encodage des chunks avec `text-embedding-ada-002` et stockage persistant dans **ChromaDB**.
3. **Retrieval + Génération** : à chaque question, on récupère les 5 chunks les plus similaires et on les injecte dans un prompt structuré pour `gpt-4o-mini`.
4. **Évaluation** : utilisation du pattern **LLM-as-a-Judge** avec `gpt-4o` pour évaluer la **Groundedness** des réponses — s'assurer que le modèle ne hallucine pas en dehors du contexte.

Ce pipeline constitue la base de toute application RAG en production.